In [1]:
# CARGAR ENTORNO DE TRABAJO
import sys
from pathlib import Path

# notebook/jupyter → proyecto/
PROJECT_ROOT = Path("../..").resolve()
sys.path.insert(0, str(PROJECT_ROOT))

from scr.python.clase_datalake import(
    ParquetStore,
    euros_taric,
    euros_taric_pais,
    euros_pais,
    kg_taric,
    kg_taric_pais,
    kg_pais,
    euros_sectores,
    euros_sectores_pais,
)

%load_ext autoreload
%autoreload 2

# Directorios
raw_base_dir = "../../data/raw"
out_dir = "../../data/rawparquet"
 
# CCAAs
# filtros_provincia = {"madrid": [28]}
filtros_provincia = {
    "nodeterminado": [0],
    "andalucia": [4, 11, 14, 18, 21, 23, 29, 41],
    "aragon": [22, 44, 50],
    "asturias": [33],
    "baleares": [7],
    "canarias": [35, 38],
    "cantabria": [39],
    "castillalamancha": [2, 13, 16, 19, 45],
    "castillayleon": [5, 9, 24, 34, 37, 40, 42, 47, 49],
    "cataluna": [8, 17, 25, 43],
    "galicia": [15, 27, 32, 36],
    "madrid": [28],
    "murcia": [30],
    "navarra": [31],
    "paisvasco": [1, 20, 48],
    "rioja": [26],
    "valencia": [3, 12, 46],
    "ceuta": [51],
    "melilla": [52]
}

# Años
ano_ini = 1995
ano_fin_def = 2023   
ano_fin = 2025       

# Mes
ultimo_mes_prov = 11

# Ámbitos y pipelinte
ambitos = ["espana"] + list(filtros_provincia.keys())
pipeline = [
    # TARIC - Euros
    ("taric", "euros_taric",                euros_taric),
    ("taric", "euros_taric_pais",           euros_taric_pais),
    
    # Totales por país
    ("sectores", "euros_pais",              euros_pais),
    ("taric", "kg_pais",                    kg_pais),
    
    # TARIC - Kilogramos
    ("taric", "kg_taric",                   kg_taric),
    ("taric", "kg_taric_pais",              kg_taric_pais),
    
    # SECTORES - Euros
    ("sectores", "euros_sectores",          euros_sectores),
    ("sectores", "euros_sectores_pais",     euros_sectores_pais)
]

In [2]:
# PROCESAR RANGO COMPLETO AUTOMÁTICAMENTE
store = ParquetStore(out_dir)

# Datos definitivos
stats_def = store.process_range(
    year_start = ano_ini,
    year_end = ano_fin_def,
    version = "def",
    raw_base_dir = raw_base_dir,
    filtros_provincia = filtros_provincia,
    skip_existing = False
)

# Datos provisionales
stats_prov = store.process_range(
    year_start = ano_fin_def + 1,
    year_end = ano_fin,
    version = "prov",
    ultimo_mes = ultimo_mes_prov,
    raw_base_dir = raw_base_dir,
    filtros_provincia = filtros_provincia,
    skip_existing = False
)

# Resumen
print("\n" + "=" * 60)
print("RESUMEN FINAL")
print("=" * 60)

print(f"\nDatos definitivos ({ano_ini}-{ano_fin_def}):")
print(f"  • Procesados: {stats_def['procesados']}")
print(f"  • Saltados: {stats_def['saltados']}")
print(f"  • Errores: {stats_def['errores']}")

print(f"\nDatos provisionales ({ano_fin_def + 1}-{ano_fin}):")
print(f"  • Procesados: {stats_prov['procesados']}")
print(f"  • Saltados: {stats_prov['saltados']}")
print(f"  • Errores: {stats_prov['errores']}")

# Cierre
store.close()
print("\n✅ Data lake completado")

2026-02-08 20:29:17 - INFO - Procesando rango 1995-2023 (def)
2026-02-08 20:29:17 - INFO - Procesando comex_taric_199501.csv
2026-02-08 20:29:18 - INFO -   ✓ Completo: taric/estado=1/anio=1995/mes=01/
2026-02-08 20:29:18 - INFO -   ✓ Nodeterminado: taric (provincias: [0])
2026-02-08 20:29:18 - INFO -   ✓ Andalucia: taric (provincias: [4, 11, 14, 18, 21, 23, 29, 41])
2026-02-08 20:29:18 - INFO -   ✓ Aragon: taric (provincias: [22, 44, 50])
2026-02-08 20:29:18 - INFO -   ✓ Asturias: taric (provincias: [33])
2026-02-08 20:29:18 - INFO -   ✓ Baleares: taric (provincias: [7])
2026-02-08 20:29:18 - INFO -   ✓ Canarias: taric (provincias: [35, 38])
2026-02-08 20:29:18 - INFO -   ✓ Cantabria: taric (provincias: [39])
2026-02-08 20:29:18 - INFO -   ✓ Castillalamancha: taric (provincias: [2, 13, 16, 19, 45])
2026-02-08 20:29:18 - INFO -   ✓ Castillayleon: taric (provincias: [5, 9, 24, 34, 37, 40, 42, 47, 49])
2026-02-08 20:29:18 - INFO -   ✓ Cataluna: taric (provincias: [8, 17, 25, 43])
2026-02-


RESUMEN FINAL

Datos definitivos (1995-2023):
  • Procesados: 348
  • Saltados: 0
  • Errores: 0

Datos provisionales (2024-2025):
  • Procesados: 23
  • Saltados: 0
  • Errores: 0

✅ Data lake completado


In [3]:
# DATALAKES DERIVADOS. RANGO COMPLETO
store = ParquetStore(out_dir)

# Datos definitivos (estado=1)
stats_def = store.process_derived_pipeline(
    pipeline = pipeline,
    ambitos = ambitos,
    year_start = ano_ini,
    year_end= ano_fin_def,
    estado = 1,
    skip_if_exists = False
)

# Datos provisionales (estado=0)
stats_prov = store.process_derived_pipeline(
    pipeline = pipeline,
    ambitos = ambitos,
    year_start = ano_fin_def + 1,
    year_end= ano_fin,
    estado = 0,
    ultimo_mes = ultimo_mes_prov,
    skip_if_exists = False
)

# Mostrar resumen
print("\n" + "="*60)
print("RESUMEN FINAL")
print("="*60)

print(f"\nDatos definitivos ({ano_ini}-{ano_fin_def}):")
print(f"  • Procesados: {stats_def['procesados']}")
print(f"  • Saltados: {stats_def['saltados']}")
print(f"  • Errores: {stats_def['errores']}")

print(f"\nDatos provisionales ({ano_fin_def + 1}-{ano_fin}):")
print(f"  • Procesados: {stats_prov['procesados']}")
print(f"  • Saltados: {stats_prov['saltados']}")
print(f"  • Errores: {stats_prov['errores']}")

print("\n✅ Pipeline completado")

2026-02-02 19:17:07 - INFO - Ejecutando pipeline: 8 transformaciones × 20 ámbitos × 29 años
2026-02-02 19:20:47 - INFO - Pipeline completado en 0:03:40.088355. Procesados: 55680, Saltados: 0, Errores: 0
2026-02-02 19:20:47 - INFO - Ejecutando pipeline: 8 transformaciones × 20 ámbitos × 2 años
2026-02-02 19:21:06 - INFO - Pipeline completado en 0:00:18.734795. Procesados: 3520, Saltados: 0, Errores: 0



RESUMEN FINAL

Datos definitivos (1995-2023):
  • Procesados: 55680
  • Saltados: 0
  • Errores: 0

Datos provisionales (2024-2025):
  • Procesados: 3520
  • Saltados: 0
  • Errores: 0

✅ Pipeline completado


In [5]:
# MES INDIVIDUAL
store = ParquetStore(out_dir)

store.csv_to_parquet(
    year = ano_fin,
    month = ultimo_mes_prov,
    version = "prov",
    raw_base_dir = raw_base_dir,
    filtros_provincia = filtros_provincia
)

print(f"\n{'='*60}")
print(f"✅ Mes {ano_fin}-{ultimo_mes_prov:02d} procesado correctamente")
print(f"{'='*60}\n")

store.close

2026-02-02 19:22:45 - INFO - Procesando comex_taric_202511.csv
2026-02-02 19:22:46 - INFO -   ✓ Completo: taric/estado=0/anio=2025/mes=11/
2026-02-02 19:22:46 - INFO -   ✓ Nodeterminado: taric (provincias: [0])
2026-02-02 19:22:46 - INFO -   ✓ Andalucia: taric (provincias: [4, 11, 14, 18, 21, 23, 29, 41])
2026-02-02 19:22:46 - INFO -   ✓ Aragon: taric (provincias: [22, 44, 50])
2026-02-02 19:22:46 - INFO -   ✓ Asturias: taric (provincias: [33])
2026-02-02 19:22:46 - INFO -   ✓ Baleares: taric (provincias: [7])
2026-02-02 19:22:46 - INFO -   ✓ Canarias: taric (provincias: [35, 38])
2026-02-02 19:22:46 - INFO -   ✓ Cantabria: taric (provincias: [39])
2026-02-02 19:22:46 - INFO -   ✓ Castillalamancha: taric (provincias: [2, 13, 16, 19, 45])
2026-02-02 19:22:46 - INFO -   ✓ Castillayleon: taric (provincias: [5, 9, 24, 34, 37, 40, 42, 47, 49])
2026-02-02 19:22:46 - INFO -   ✓ Cataluna: taric (provincias: [8, 17, 25, 43])
2026-02-02 19:22:46 - INFO -   ✓ Galicia: taric (provincias: [15, 27, 


✅ Mes 2025-11 procesado correctamente



<bound method ParquetStore.close of <scr.python.clase_datalake.ParquetStore object at 0x7f109c103890>>

In [6]:
# MES INDIVIDUAL. DATALAKES DERIVADOS
store = ParquetStore(out_dir)

stats = store.process_month_pipeline(
    year = ano_fin,
    month = ultimo_mes_prov,
    estado = 0, 
    pipeline = pipeline,
    ambitos = ambitos,
    skip_if_exists = False
)

store.close()

print(f"\n✅ Pipeline completado")
print(f"Procesados: {stats['procesados']}")
print(f"Errores: {stats['errores']}")

2026-02-02 19:22:55 - INFO - ============================================================
2026-02-02 19:22:55 - INFO - Procesando pipeline mes: 2025-11 (prov)
2026-02-02 19:22:55 - INFO - ============================================================
2026-02-02 19:22:55 - INFO - Pipeline: 8 transformaciones
2026-02-02 19:22:55 - INFO - Ámbitos: espana, nodeterminado, andalucia, aragon, asturias, baleares, canarias, cantabria, castillalamancha, castillayleon, cataluna, galicia, madrid, murcia, navarra, paisvasco, rioja, valencia, ceuta, melilla
2026-02-02 19:22:55 - INFO - 
📍 Ámbito: espana
2026-02-02 19:22:55 - INFO - 
📍 Ámbito: nodeterminado
2026-02-02 19:22:55 - INFO - 
📍 Ámbito: andalucia
2026-02-02 19:22:55 - INFO - 
📍 Ámbito: aragon
2026-02-02 19:22:55 - INFO - 
📍 Ámbito: asturias
2026-02-02 19:22:55 - INFO - 
📍 Ámbito: baleares
2026-02-02 19:22:55 - INFO - 
📍 Ámbito: canarias
2026-02-02 19:22:55 - INFO - 
📍 Ámbito: cantabria
2026-02-02 19:22:55 - INFO - 
📍 Ámbito: castillalamancha



✅ Pipeline completado
Procesados: 160
Errores: 0


In [ ]:
# AÑO INDIVIDUAL
store = ParquetStore(out_dir)

stats_def = store.process_range(
    year_start = ano_fin_def,
    year_end = ano_fin_def,
    version = "def",
    raw_base_dir = raw_base_dir,
    filtros_provincia = filtros_provincia,
    skip_existing = False
)

print(f"\n{'='*60}")
print(f"Resumen año {ano_fin_def}:")
print(f"  • Procesados: {stats_def['procesados']}")
print(f"  • Saltados: {stats_def['saltados']}")
print(f"  • Errores: {stats_def['errores']}")
print(f"\n✅ Año {ano_fin_def} procesado correctamente")
print(f"{'='*60}\n")

store.close

In [ ]:
# AÑO INDIVIDUAL. DATALAKE DERIVADO
store = ParquetStore(out_dir)

stats_def = store.process_derived_pipeline(
    pipeline = pipeline,
    ambitos = ambitos,
    year_start = ano_fin_def,
    year_end= ano_fin_def,
    estado = 1,
    skip_if_exists = False
)

print(f"\n{'='*60}")
print(f"Resumen mes {ano_fin}-{ultimo_mes_prov:02d}:")
print(f"  • Procesados: {stats['procesados']}")
print(f"  • Saltados: {stats['saltados']}")
print(f"  • Errores: {stats['errores']}")
print(f"\n✅ Pipeline completado para {ano_fin}-{ultimo_mes_prov:02d}")
print(f"{'='*60}\n")